### **Tópico 1: Instalação de Dependências**

Esta é a única seção obrigatória que prepara o ambiente do Colab. Ela instala o Conda, cria um ambiente isolado chamado `ns` e instala todas as bibliotecas necessárias, como PyTorch, Nerfstudio e outras ferramentas. Execute esta célula uma vez no início de cada sessão. O kernel será reiniciado ao final.

In [ ]:
# Célula 1.1: Instala o Conda no ambiente do Google Colab
# Isso reiniciará o kernel automaticamente. Após o reinício, prossiga para a célula 1.2.
!pip -q install condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:06
🔁 Restarting kernel...


### **Tópico 2: Configuração de Diretórios (Google Drive)**

Esta seção monta seu Google Drive e cria a estrutura de pastas padrão para organizar seus dados e resultados. É altamente recomendado usar o Drive para que seus arquivos não sejam perdidos ao final da sessão do Colab.

In [ ]:
# Célula 1.2: Cria o ambiente e instala as dependências do Nerfstudio
# Este passo pode demorar alguns minutos.

# Cria um ambiente isolado chamado "ns" com Python 3.10
!mamba create -y -n ns python=3.10

# Instala PyTorch com suporte a CUDA 11.8
!mamba install -y -n ns -c pytorch -c nvidia pytorch torchvision pytorch-cuda=11.8

# Instala COLMAP (para processamento de dados), ffmpeg (para vídeos) e git
!mamba install -y -n ns -c conda-forge colmap ffmpeg git openh264

# Instala Nerfstudio e a métrica LPIPS via pip dentro do ambiente "ns"
!conda run -n ns python -m pip install -U pip wheel setuptools
!conda run -n ns python -m pip install nerfstudio lpips

# Instala utilitários do sistema para visualização remota e execução headless
# Localtunnel: Expõe a porta do viewer para acesso via web
# Xvfb: Servidor de display virtual, necessário para rodar o COLMAP sem uma GUI no Colab
!apt-get -qq update && apt-get -qq install -y nodejs npm xvfb

!npm -g -s install localtunnel

# Define um prefixo para garantir que todos os comandos do Nerfstudio rodem no ambiente correto
NS_PREFIX = "conda run -n ns"


Looking for: ['python=3.10']

[+] 0.0s
conda-forge/linux-64  ⣾  [+] 0.1s
conda-forge/linux-64  ⣾  
conda-forge/noarch    ⣾  [+] 0.2s
conda-forge/linux-64   4%
conda-forge/noarch     3%[+] 0.3s
conda-forge/linux-64  14%
conda-forge/noarch    23%[+] 0.4s
conda-forge/linux-64  25%
conda-forge/noarch    45%[+] 0.5s
conda-forge/linux-64  35%
conda-forge/noarch    66%[+] 0.6s
conda-forge/linux-64  45%
conda-forge/noarch    86%[+] 0.7s
conda-forge/linux-64  50%
conda-forge/noarch    97%conda-forge/noarch                                
[+] 0.8s
conda-forge/linux-64  52%[+] 0.9s
conda-forge/linux-64  71%[+] 1.0s
conda-forge/linux-64  91%[+] 1.1s
conda-forge/linux-64 100%[+] 1.2s
conda-forge/linux-64 100%conda-forge/linux-64                              
Transaction

  Prefix: /usr/local/envs/ns

  Updating specs:

   - python=3.10


  Package               Version  Build                 Channel           Size
───────────────────────────────────────────────────────────────────────────────
  In

In [ ]:
# Verificar GPUs

!nvidia-smi

!nvcc --version

Wed Mar  4 00:25:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Célula 2.1: Monta o Google Drive e define a estrutura de pastas

from google.colab import drive
import os

# Monta o Google Drive no diretório /content/drive
drive.mount('/content/drive')

# --- CONFIGURAÇÃO ---
# Altere o caminho base se desejar organizar seus projetos de outra forma.
DIRETORIO_BASE = "/content/drive/MyDrive/Faculdade/TCC/desenvolvimento"

# Define os caminhos para cada tipo de dado
DATA_DIR    = os.path.join(DIRETORIO_BASE, "data")       # Imagens/vídeos de entrada
DATASET_DIR = os.path.join(DIRETORIO_BASE, "datasets")   # Datasets processados pelo COLMAP
OUTPUTS_DIR = os.path.join(DIRETORIO_BASE, "outputs")    # Resultados do treinamento (checkpoints)
EXPORTS_DIR = os.path.join(DIRETORIO_BASE, "exports")    # Arquivos exportados (renders, .ply)

# Define o diretório final no Google Drive
CAMINHO_DATASET = os.path.join(DATASET_DIR, NOME_DA_CENA)
os.makedirs(CAMINHO_DATASET, exist_ok=True)

# Cria os diretórios se eles não existirem
for path in [DATA_DIR, DATASET_DIR, OUTPUTS_DIR, EXPORTS_DIR]:
    os.makedirs(path, exist_ok=True)
    print(f"Diretório garantido: {path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


NameError: name 'NOME_DA_CENA' is not defined

### **Tópico 3: Aquisição de Dados de Entrada**

Escolha **uma** das opções abaixo para definir qual cena você irá processar. Você pode usar um dos datasets de exemplo do Nerfstudio, fazer o upload de suas próprias imagens ou de um vídeo.

In [ ]:
# Célula 3.1: Escolha a fonte dos seus dados

import os
from google.colab import files

# --- CONFIGURAÇÃO ---
# Escolha uma opção: 'poster', 'dozer', 'desolation', 'UPLOAD_IMAGENS', ou 'UPLOAD_VIDEO'
FONTE_DOS_DADOS = "poster"

# Define o nome da cena com base na opção
if FONTE_DOS_DADOS in ["poster", "dozer", "desolation"]:
    NOME_DA_CENA = FONTE_DOS_DADOS
    # Baixa e descompacta o dataset de exemplo
    !{NS_PREFIX} ns-download-data nerfstudio --capture-name={NOME_DA_CENA}
    # O caminho de origem será o diretório de dados do Nerfstudio
    CAMINHO_ORIGEM = f"/content/data/nerfstudio/{NOME_DA_CENA}"

elif FONTE_DOS_DADOS == "UPLOAD_IMAGENS":
    NOME_DA_CENA = "dados_customizados"
    # Define um subdiretório para as imagens brutas
    CAMINHO_ORIGEM = os.path.join(DATA_DIR, NOME_DA_CENA, "images_raw")
    os.makedirs(CAMINHO_ORIGEM, exist_ok=True)
    print(f"Faça o upload de suas imagens (.jpg, .png) para o diretório: {CAMINHO_ORIGEM}")
    # Abre a janela de upload do Colab
    uploaded = files.upload()
    for filename, content in uploaded.items():
        with open(os.path.join(CAMINHO_ORIGEM, filename), 'wb') as f:
            f.write(content)
    print(f"{len(uploaded)} imagens salvas em {CAMINHO_ORIGEM}")

elif FONTE_DOS_DADOS == "UPLOAD_VIDEO":
    NOME_DA_CENA = "dados_customizados"
    VIDEO_DIR = os.path.join(DATA_DIR, NOME_DA_CENA)
    os.makedirs(VIDEO_DIR, exist_ok=True)
    print("Faça o upload do seu arquivo de vídeo (.mp4, .mov, etc.)")
    uploaded = files.upload()
    # Pega o nome do primeiro arquivo enviado
    video_filename = list(uploaded.keys())[0]
    CAMINHO_ORIGEM = os.path.join(VIDEO_DIR, video_filename)
    os.rename(video_filename, CAMINHO_ORIGEM)

else:
    raise ValueError("FONTE_DOS_DADOS inválida. Verifique as opções disponíveis.")

print(f"\nFonte de dados definida. Nome da cena: '{NOME_DA_CENA}'")
print(f"Caminho de origem: {CAMINHO_ORIGEM}")

Downloading...
From (original): https://drive.google.com/uc?id=1FceQ5DX7bbTbHeL26t0x6ku56cwsRs6t
From (redirected): https://drive.google.com/uc?id=1FceQ5DX7bbTbHeL26t0x6ku56cwsRs6t&confirm=t&uuid=cec88e86-5c39-4c59-9e27-0a50c4452ae7
To: /content/data/nerfstudio/poster.zip

  0%|          | 0.00/749M [00:00<?, ?B/s]
  1%|          | 4.72M/749M [00:00<00:26, 28.6MB/s]
  1%|▏         | 11.0M/749M [00:00<00:17, 41.1MB/s]
  4%|▍         | 29.4M/749M [00:00<00:07, 93.4MB/s]
  5%|▌         | 39.8M/749M [00:00<00:11, 61.3MB/s]
  8%|▊         | 59.2M/749M [00:00<00:07, 90.6MB/s]
  9%|▉         | 70.8M/749M [00:00<00:08, 78.7MB/s]
 12%|█▏        | 88.6M/749M [00:01<00:06, 101MB/s] 
 14%|█▎        | 101M/749M [00:01<00:06, 101MB/s] 
 15%|█▌        | 113M/749M [00:01<00:06, 103MB/s]
 17%|█▋        | 124M/749M [00:01<00:07, 88.1MB/s]
 19%|█▉        | 141M/749M [00:01<00:07, 83.8MB/s]
 22%|██▏       | 167M/749M [00:01<00:04, 120MB/s] 
 24%|██▍       | 181M/749M [00:01<00:04, 120MB/s]
 26%|██▌       

### **Tópico 4: (Opcional) Redimensionamento de Imagens**

Se você fez o upload de imagens de alta resolução (`UPLOAD_IMAGENS`), executá-las pode consumir muita memória e tempo. Esta etapa opcional redimensiona as imagens para uma dimensão máxima, mantendo a proporção.

In [ ]:
# Célula 4.1: Redimensiona imagens para acelerar o processamento

import os
from PIL import Image
from tqdm import tqdm

# --- CONFIGURAÇÃO ---
EXECUTAR_REDIMENSIONAMENTO = True # Mude para True se desejar redimensionar
MAX_DIMENSAO = 1600 # Dimensão máxima para a maior borda da imagem (largura ou altura)

if EXECUTAR_REDIMENSIONAMENTO:
    if FONTE_DOS_DADOS != "UPLOAD_IMAGENS":
        print("Aviso: O redimensionamento só é útil para a opção 'UPLOAD_IMAGENS'. Pulando.")
    else:
        # Diretório de destino para imagens redimensionadas
        dest_dir = os.path.join(DATA_DIR, NOME_DA_CENA, "images")
        os.makedirs(dest_dir, exist_ok=True)

        image_files = [f for f in os.listdir(CAMINHO_ORIGEM) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        print(f"Encontradas {len(image_files)} imagens para redimensionar...")

        for filename in tqdm(image_files, desc="Redimensionando Imagens"):
            try:
                with Image.open(os.path.join(CAMINHO_ORIGEM, filename)) as img:
                    width, height = img.size
                    if width > MAX_DIMENSAO or height > MAX_DIMENSAO:
                        if width > height:
                            new_width = MAX_DIMENSAO
                            new_height = int(height * (MAX_DIMENSAO / width))
                        else:
                            new_height = MAX_DIMENSAO
                            new_width = int(width * (MAX_DIMENSAO / height))

                        # Usa o filtro LANCZOS para alta qualidade de redimensionamento
                        img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)

                    img.save(os.path.join(dest_dir, filename))
            except Exception as e:
                print(f"Não foi possível processar {filename}: {e}")

        # Atualiza o caminho de origem para que o próximo passo use as imagens redimensionadas
        CAMINHO_ORIGEM = dest_dir
        print(f"\nRedimensionamento concluído. Novo caminho de origem para processamento: {CAMINHO_ORIGEM}")
else:
    print("Redimensionamento de imagens pulado.")

Aviso: O redimensionamento só é útil para a opção 'UPLOAD_IMAGENS'. Pulando.


### **Tópico 5: Processamento de Dados (Execução do COLMAP)**

Este tópico pega as imagens ou o vídeo do `CAMINHO_ORIGEM` e usa o COLMAP para gerar as poses da câmera e a nuvem de pontos esparsa, salvando tudo em um formato que o Nerfstudio entende.

In [ ]:
# Célula 5.1: Executa o processamento de dados do Nerfstudio

import os
import shutil
import subprocess
from distutils.dir_util import copy_tree

# Força o Qt a rodar sem interface gráfica
os.environ["QT_QPA_PLATFORM"] = "offscreen"

# --- OTIMIZAÇÃO: USAR DISCO LOCAL ---
# Define um diretório temporário local para processamento rápido
TMP_BASE = "/content/tmp_ns"
TMP_OUT = os.path.join(TMP_BASE, NOME_DA_CENA)

# Garante que o diretório temporário esteja limpo
if os.path.exists(TMP_OUT):
    shutil.rmtree(TMP_OUT)
os.makedirs(TMP_OUT, exist_ok=True)

LOG_FILE = os.path.join(CAMINHO_DATASET, "processing.log")

print(f"Iniciando o processamento.")
print(f"Origem: {CAMINHO_ORIGEM}")
print(f"Processando temporariamente em: {TMP_OUT} (SSD Local)")
print(f"Destino Final: {CAMINHO_DATASET} (Google Drive)")
print(f"Um log detalhado será salvo em: {LOG_FILE}")

# Verifica se a origem é um diretório (imagens) ou um arquivo (vídeo)
if os.path.isdir(CAMINHO_ORIGEM):
    # Comando para processar um diretório de imagens
    # Adicionada flag --no-gpu para contornar crash do OpenGL/SiftGPU
    cmd_str = f"""
    xvfb-run -s "-screen 0 3840x2160x24" -a {NS_PREFIX} ns-process-data images \
        --data "{CAMINHO_ORIGEM}" \
        --output-dir "{TMP_OUT}" \
        --matching-method sequential \
        --no-gpu
    """
else:
    # Comando para processar um arquivo de vídeo
    cmd_str = f"""
    xvfb-run -s "-screen 0 3840x2160x24" -a {NS_PREFIX} ns-process-data video \
        --data "{CAMINHO_ORIGEM}" \
        --output-dir "{TMP_OUT}" \
        --num-frames-to-sample 300 \
        --matching-method sequential \
        --no-gpu
    """

# Limpa espaços extras e quebras de linha do comando
cmd_clean = " ".join(cmd_str.split())

print("Executando comando...")

# Executa o comando usando subprocess para capturar erros
with open(LOG_FILE, "w") as f:
    try:
        # check=True lança uma exceção se o código de retorno for diferente de 0
        subprocess.run(cmd_clean, shell=True, check=True, stdout=f, stderr=subprocess.STDOUT)
    except subprocess.CalledProcessError as e:
        print(f"\nERRO CRÍTICO: O comando falhou com código de retorno {e.returncode}.")
        print(f"Verifique o log para detalhes: {LOG_FILE}")
        # Levanta a exceção novamente para parar a execução da célula
        raise e

print("Processamento local concluído com sucesso. Copiando arquivos para o Google Drive...")

# Copia os resultados do SSD local para o Google Drive
try:
    copy_tree(TMP_OUT, CAMINHO_DATASET)
    print(f"Sucesso! Dataset sincronizado em: {CAMINHO_DATASET}")
    # Opcional: Limpar tmp para economizar espaço se necessário
    # shutil.rmtree(TMP_OUT)
except Exception as e:
    print(f"Erro ao copiar arquivos para o Drive: {e}")

Iniciando o processamento.
Origem: /content/data/nerfstudio/poster
Processando temporariamente em: /content/tmp_ns/poster (SSD Local)
Destino Final: /content/drive/MyDrive/Faculdade/TCC/desenvolvimento/datasets/poster (Google Drive)
Um log detalhado será salvo em: /content/drive/MyDrive/Faculdade/TCC/desenvolvimento/datasets/poster/processing.log
Executando comando...
Processamento local concluído com sucesso. Copiando arquivos para o Google Drive...
Sucesso! Dataset sincronizado em: /content/drive/MyDrive/Faculdade/TCC/desenvolvimento/datasets/poster


In [ ]:
# Célula 5.2: (Opcional) Visualizar o final do log de processamento para verificar erros

!tail -n 30 "{LOG_FILE}"

[23:53:41] 🎉 Done copying images with prefix 'frame_'.                                        process_data_utils.py:348
[00:03:08] 🎉 Done extracting COLMAP features.                                                       colmap_utils.py:137
[00:16:13] 🎉 Done matching COLMAP features.                                                         colmap_utils.py:151
[00:34:41] 🎉 Done COLMAP bundle adjustment.                                                         colmap_utils.py:173
[00:36:20] 🎉 Done refining intrinsics.                                                              colmap_utils.py:184
[00:36:24] 🎉 🎉 🎉 All DONE 🎉 🎉 🎉                                                images_to_nerfstudio_dataset.py:135
           Starting with 904 images                                                  images_to_nerfstudio_dataset.py:138
           Colmap matched 904 images                                                 images_to_nerfstudio_dataset.py:138
           COLMAP found poses for all image

# DAQUI PRA BAIXO TA RUIM, só BO

### **Tópico 6: Treinamento do Modelo**

Agora é hora de treinar o modelo. Primeiro, vamos configurar o `localtunnel` para que você possa acompanhar o progresso em tempo real no viewer web. Depois, escolha **uma** das células de treinamento: `nerfacto` (focado em qualidade) ou `splatfacto` (focado em velocidade e exportação para Gaussian Splatting).

In [ ]:
# Célula 6.1: Configura o viewer remoto
import time

# Garante que qualquer túnel anterior seja fechado
!pkill -f localtunnel

# Abre um novo túnel na porta 7007 e envia a saída para um arquivo
get_ipython().system_raw('lt --port 7007 > url.txt 2>&1 &')
time.sleep(3) # Espera um pouco para o túnel ser estabelecido

try:
    with open('url.txt', 'r') as f:
        lt_url = f.read().strip().split('your url is: ')[-1]

    # Formata a URL para o viewer do Nerfstudio
    if lt_url:
        websocket_url = lt_url.replace("https://", "wss://")
        viewer_url = f"https://viewer.nerf.studio/?websocket_url={websocket_url}"
        print("Viewer pronto! Acesse o link abaixo para acompanhar o treino:")
        print(viewer_url)
    else:
        print("Falha ao criar o túnel. O log do localtunnel é:")
        !cat url.txt
except FileNotFoundError:
    print("Arquivo url.txt não encontrado. Não foi possível obter a URL do viewer.")

Viewer pronto! Acesse o link abaixo para acompanhar o treino:
https://viewer.nerf.studio/?websocket_url=wss://fluffy-cycles-enjoy.loca.lt


In [ ]:
# Célula 6.2: Treinar com `nerfacto` (Modelo de NeRF balanceado)

# O treinamento continuará até que você o interrompa manualmente ou por 30.000 passos (padrão).
# O parâmetro --viewer.quit-on-train-completion=True fecha o viewer ao final.

# FIXME: Por algum motivo isso não funciona com o {NS_PREFIX}
# !{NS_PREFIX} ns-train nerfacto \
#   --data "{CAMINHO_DATASET}" \
#   --output-dir "{OUTPUTS_DIR}" \
#   --viewer.quit-on-train-completion True \
#   --viewer.websocket-port 7007

cmd = (
    f'conda run -n ns ns-train nerfacto '
    f'--data "{CAMINHO_DATASET}" '
    f'--output-dir "{OUTPUTS_DIR}" '
    f'--viewer.quit-on-train-completion True '
    f'--viewer.websocket-port 7007'
)
print(cmd)
!{cmd}

NameError: name 'CAMINHO_DATASET' is not defined

In [ ]:
# Célula 6.3: Treinar com `splatfacto` (Modelo rápido de Gaussian Splatting)

# Este modelo é muito mais rápido e ideal para visualização em tempo real e exportação de .ply.
# O parâmetro --pipeline.datamanager.cache-images=disk ajuda a economizar RAM.

!{NS_PREFIX} ns-train splatfacto \
    --data "{CAMINHO_DATASET}" \
    --output-dir "{OUTPUTS_DIR}" \
    --viewer.quit-on-train-completion True \
    --viewer.websocket-port 7007 \
    --pipeline.datamanager.cache-images disk

/bin/bash: line 1: {NS_PREFIX}: command not found


### **Tópico 7: Exportação e Geração de Artefatos**

Após o treinamento, você pode usar o modelo treinado para gerar diferentes saídas, como uma nuvem de pontos `Gaussian Splat` (se usou `splatfacto`) ou renderizar um vídeo de uma trajetória de câmera.

In [ ]:
# Célula 7.1: Encontra o arquivo de configuração do último treinamento realizado

import glob
import os

# Busca recursivamente por todos os arquivos config.yml no diretório de saídas
configs_encontrados = sorted(
    glob.glob(os.path.join(OUTPUTS_DIR, "**", "config.yml"), recursive=True),
    key=os.path.getmtime
)

if not configs_encontrados:
    raise FileNotFoundError("Nenhum arquivo 'config.yml' de treinamento foi encontrado.")

# Pega o arquivo mais recente
CAMINHO_CONFIG_TREINO = configs_encontrados[-1]
print(f"Usando o arquivo de configuração do treinamento mais recente:\n{CAMINHO_CONFIG_TREINO}")

In [ ]:
# Célula 7.2: Exporta a nuvem de pontos Gaussian Splat (.ply)

import yaml

# Verifica se o método de treinamento foi splatfacto antes de exportar
with open(CAMINHO_CONFIG_TREINO, 'r') as f:
    config_data = yaml.safe_load(f)

if config_data.get("_target").endswith("splatfacto"):
    print("Modelo 'splatfacto' detectado. Exportando Gaussian Splat...")

    CAMINHO_EXPORT_GS = os.path.join(EXPORTS_DIR, NOME_DA_CENA, "gaussian_splat")
    os.makedirs(CAMINHO_EXPORT_GS, exist_ok=True)

    !{NS_PREFIX} ns-export gaussian-splat \\
        --load-config "{CAMINHO_CONFIG_TREINO}" \\
        --output-dir "{CAMINHO_EXPORT_GS}"

    print(f"\nExportação concluída! Arquivo .ply salvo em: {CAMINHO_EXPORT_GS}")
else:
    print("Aviso: O modelo treinado não é 'splatfacto'. Pulando a exportação de Gaussian Splat.")

In [ ]:
# Célula 7.3: Renderiza um vídeo com uma trajetória de câmera em espiral

CAMINHO_EXPORT_VIDEO = os.path.join(EXPORTS_DIR, NOME_DA_CENA, "renders")
os.makedirs(CAMINHO_EXPORT_VIDEO, exist_ok=True)
VIDEO_PATH = os.path.join(CAMINHO_EXPORT_VIDEO, "render_spiral.mp4")

print(f"Iniciando a renderização do vídeo. Isso pode demorar...")

!{NS_PREFIX} ns-render camera-path \\
    --load-config "{CAMINHO_CONFIG_TREINO}" \\
    --camera-path-filename "{CAMINHO_DATASET}/camera_path.json" \\
    --output-path "{VIDEO_PATH}"

print(f"\nVídeo renderizado e salvo em: {VIDEO_PATH}")

In [ ]:
# Célula 7.4: Compacta todos os arquivos exportados em um único .zip para download

import zipfile
import os

CAMINHO_BASE_EXPORT = os.path.join(EXPORTS_DIR, NOME_DA_CENA)
CAMINHO_ZIP = f"{CAMINHO_BASE_EXPORT}_exports.zip"

print(f"Criando arquivo zip em: {CAMINHO_ZIP}")

with zipfile.ZipFile(CAMINHO_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(CAMINHO_BASE_EXPORT):
        for filename in files:
            file_path = os.path.join(root, filename)
            # O segundo argumento do zf.write é o caminho dentro do zip
            zf.write(file_path, os.path.relpath(file_path, CAMINHO_BASE_EXPORT))

print("Compactação concluída!")